#### **Biostars -- TAG**

In [1]:
#!pip install requests beautifulsoup4

In [2]:
import re, time, json, csv, html
from pathlib import Path
from urllib.parse import urlencode
from functools import lru_cache
import requests
from bs4 import BeautifulSoup
from collections import defaultdict

In [6]:
# ========================
# Paths & output
# ========================
TAGS_PATH  = Path("../02-datasource/biostars_tags.json")
OUT_DIR    = Path("../04-output/raw/biostars")
OUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH = OUT_DIR / "biostars_metagenomics_tag.jsonl"
CSV_PATH   = OUT_DIR / "biostars_metagenomics_tag.csv"
CACHE_IDS  = OUT_DIR / "seen_ids_tag.txt"

HEADERS = {
    "User-Agent": "MetagenomicsCollector/1.3 (+research; contact: mahouzonssou.akotenou@um6p.ma)"
}

# ========================
# Crawl config
# ========================
SLEEP_BETWEEN_REQUESTS = 0.200  # ~200 ms between requests

# Order variants to fetch per tag (exactly one page each)
# - 'creation'   : newest first (recent)
# - 'rank'       : most interesting
# - 'votes'      : most upvoted
# - 'replies'    : most replied
ORDER_VARIANTS = ["creation", "rank", "votes", "replies"]


In [7]:
# ========================
# Small helpers
# ========================
def sleep():
    time.sleep(SLEEP_BETWEEN_REQUESTS)

def load_seen_ids():
    if CACHE_IDS.exists():
        return set(x.strip() for x in CACHE_IDS.read_text().splitlines() if x.strip())
    return set()

def save_seen_ids(ids):
    with open(CACHE_IDS, "w", encoding="utf-8") as f:
        for pid in sorted(ids, key=lambda x: int(x)):
            f.write(f"{pid}\n")

def fetch(url, as_json=False, params=None):
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return r.json() if as_json else r.text

def html_plain_text(xhtml):
    try:
        return BeautifulSoup(xhtml or "", "html.parser").get_text("\n", strip=True)
    except Exception:
        return (xhtml or "").strip()

def as_int_or_str(x):
    try:
        return int(x)
    except Exception:
        return str(x)

def normalize_tags(tag_val, fallback=None):
    # API returns space-separated string; sometimes list
    if isinstance(tag_val, list):
        return [t for t in tag_val if t]
    if isinstance(tag_val, str):
        toks = [t.strip() for t in tag_val.split() if t.strip()]
        if toks:
            return toks
    return list(fallback) if fallback else []

def get_body_text(post_json: dict) -> str:
    # Biostars uses 'xhtml' or 'content'
    html_blob = post_json.get("xhtml") or post_json.get("content") or ""
    return html_plain_text(html_blob)

# Keep everything in tag-only mode
def apply_labels_from_config(_text): return []
def blocked_by_global_excludes(_text): return False

# ========================
# Read list of tags
# ========================
def load_tags_list(path: Path):
    if path.exists():
        try:
            tags = json.loads(path.read_text(encoding="utf-8"))
            tags = [t.strip() for t in tags if isinstance(t, str) and t.strip()]
            if tags:
                return tags
        except Exception as e:
            print(f"[warn] failed to read {path}: {e}")
    # fallback to your example list
    return ["rna-seq", "metagenomics", "microbiome", "shotgun-metagenomics", "16s", "assembly", "binning"]

TAGS = load_tags_list(TAGS_PATH)
print(f"Loaded {len(TAGS)} tags.")

# ========================
# Tag URLs (single page per order)
# ========================
def tag_order_urls(tag):
    base = f"https://www.biostars.org/tag/{tag}/"
    for order in ORDER_VARIANTS:
        yield f"{base}?order={order}", order

def extract_post_ids_from_index(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    ids = set()
    for a in soup.find_all("a", href=True):
        m = re.search(r"/p/(\d+)/?", a["href"])
        if m:
            ids.add(m.group(1))
    return ids

Loaded 1 tags.


In [8]:
# ========================
# API helpers
# ========================
@lru_cache(maxsize=8192)
def get_post_json(pid):
    return fetch(f"https://www.biostars.org/api/post/{pid}/", as_json=True)

@lru_cache(maxsize=4096)
def get_user_json(uid):
    """Fetch Biostars user profile once; cached."""
    if uid in (None, ""):
        return {}
    try:
        return fetch(f"https://www.biostars.org/api/user/{uid}/", as_json=True)
    except Exception:
        return {}

def compact_user_profile(u):
    if not u:
        return None
    return {
        "id": u.get("id"),
        "name": u.get("name"),
        "date_joined": u.get("date_joined"),
        "last_login": u.get("last_login"),
        "joined_days_ago": u.get("joined_days_ago"),
        # votes GIVEN by the user
        "vote_count": u.get("vote_count"),
    }

def make_author_obj(name, uid):
    """Return merged author object with profile fields (uid may be str)."""
    if uid in (None, ""):
        return {
            "id": None, "name": name, "date_joined": None,
            "last_login": None, "vote_count": None, "joined_days_ago": None
        }
    uid_norm = as_int_or_str(uid)
    prof_raw = compact_user_profile(get_user_json(uid_norm)) or {}
    return {
        "id": uid_norm,
        "name": name,
        "date_joined": prof_raw.get("date_joined"),
        "last_login": prof_raw.get("last_login"),
        "joined_days_ago": prof_raw.get("joined_days_ago"),
        "vote_count": prof_raw.get("vote_count"),
    }

# ========================
# HTML: enumerate answers & accepted
# ========================
def extract_answer_ids_and_accept_flag(soup):
    """
    Return (answer_ids, accepted_id) from a Biostars thread HTML.
    - Each answer is under: div.post.view.open[data-value]
    - Accepted answer body has itemprop="acceptedAnswer"
    - Other answers use itemprop="suggestedAnswer"
    """
    ans_ids, accepted_id = [], None
    for block in soup.select('div.post.view.open[data-value]'):
        pid = block.get("data-value")
        if not pid:
            continue
        body = block.select_one(
            'div.body[itemprop="acceptedAnswer"], div.body[itemprop="suggestedAnswer"]'
        )
        if body:
            ans_ids.append(pid)
            if body.get("itemprop") == "acceptedAnswer":
                accepted_id = pid
    return ans_ids, accepted_id

# ========================
# Thread parsing (API + HTML) incl. user profiles
# ========================
def parse_thread_page(pid):
    """
    - Use HTML to enumerate answer IDs and detect accepted answer.
    - Use API to get clean bodies & metadata.
    - Merge author (id/name + profile fields) into a single object.
    """
    url = f"https://www.biostars.org/p/{pid}/"
    soup = BeautifulSoup(fetch(url), "html.parser")

    # Title
    title_el = soup.select_one('div.post.view .title[itemprop="name"]') or soup.select_one("div.title")
    title = title_el.get_text(strip=True) if title_el else ""

    # Question via API
    q_json = get_post_json(pid)
    q_text = get_body_text(q_json) or ""
    if not q_text.strip():
        q_html_el = soup.select_one('[itemprop="mainEntity"] [itemprop="text"]')
        if q_html_el:
            q_text = q_html_el.get_text("\n", strip=True)

    q_created_at  = q_json.get("creation_date")
    q_author_uid  = q_json.get("author_uid") or q_json.get("author_id")
    q_author_name = q_json.get("author")
    q_author      = make_author_obj(q_author_name, q_author_uid)

    # Tags
    tags_api  = normalize_tags(q_json.get("tag_val"))
    tags_html = [a.get_text(strip=True) for a in soup.select("span.inplace-tags a.ptag")]

    # Answers
    ans_ids, accepted_id = extract_answer_ids_and_accept_flag(soup)
    answers = []
    for aid in ans_ids:
        a_json = get_post_json(aid)
        a_text = get_body_text(a_json)
        if not a_text or len(a_text) < 20:
            continue

        a_author_uid  = a_json.get("author_uid") or a_json.get("author_id")
        a_author_name = a_json.get("author")
        a_author      = make_author_obj(a_author_name, a_author_uid)

        answers.append({
            "answer_id": as_int_or_str(aid),
            "text": a_text,
            "score": int(a_json.get("vote_count", 0) or 0),
            "accepted": (str(aid) == str(accepted_id)),
            "created_at": a_json.get("creation_date"),
            "author": a_author,
        })

    # Pick best: accepted > highest score > longer
    best = None
    if answers:
        acc = [a for a in answers if a["accepted"]]
        best = (sorted(acc, key=lambda x: (x["score"], len(x["text"])), reverse=True)[0]
                if acc else sorted(answers, key=lambda x: (x["score"], len(x["text"])), reverse=True)[0])

    return {
        "url": url,
        "title": title,
        "question_text": q_text,
        "question_created_at": q_created_at,
        "question_author": q_author,           # merged object
        "answers": answers,                    # each with merged author
        "best_answer": best,
        "tags_api": tags_api,
        "tags_html": tags_html,
    }

# ========================
# Normalizer (adds answers_all, origin tag + order)
# ========================
def normalize_record(pid, thread_meta, post_json, origin_tag="", order_variant=""):
    q = (thread_meta.get("question_text") or "").strip()
    a_best = (thread_meta.get("best_answer", {}) or {}).get("text", "").strip()
    if not q or not a_best:
        return None

    tags_site = normalize_tags(
        post_json.get("tag_val"),
        fallback=(thread_meta.get("tags_api") or thread_meta.get("tags_html") or [])
    )

    full_text = " ".join([thread_meta.get("title",""), q, a_best, " ".join(tags_site)])
    if blocked_by_global_excludes(full_text):
        return None
    labels = apply_labels_from_config(full_text)

    return {
        "source": "biostars",
        "post_id": as_int_or_str(pid),
        "url": thread_meta["url"],
        "title": thread_meta.get("title", ""),
        "search_query": origin_tag,           # keep legacy field name
        "crawl_tag": origin_tag,              # explicit
        "crawl_order": order_variant,         # NEW: which order produced this

        # Question (author merged)
        "question": q,
        "question_created_at": thread_meta.get("question_created_at"),
        "question_author": thread_meta.get("question_author"),

        # Answers
        "answer": a_best,
        "answers_all": thread_meta.get("answers", []),
        "accepted": bool(thread_meta.get("best_answer", {}).get("accepted", False)),
        "score": int(thread_meta.get("best_answer", {}).get("score", 0) or 0),

        # Site/meta
        "tags_site": tags_site,
        "created_at": post_json.get("creation_date"),
        "has_accepted": post_json.get("has_accepted", False),
        "view_count": post_json.get("view_count", 0),

        # Auto-labels (empty)
        "topic_labels": labels,
    }

In [9]:
# ========================
# Crawl (by tag, one page per order)
# ========================
from collections import defaultdict

seen = load_seen_ids()
collected_total = 0
drop_stats = defaultdict(int)

with open(JSONL_PATH, "a", encoding="utf-8") as jf:
    for tag in TAGS:
        print(f"\n=== TAG: {tag} ===")
        term_collected = 0
        for url, order_name in tag_order_urls(tag):
            try:
                html_text = fetch(url)
            except Exception as e:
                print("Tag page fetch failed:", url, e)
                sleep(); continue
            sleep()

            pids = extract_post_ids_from_index(html_text)
            if not pids:
                print(f"[{tag}|{order_name}] no post ids on {url}")
                continue

            print(f"[{tag}|{order_name}] {len(pids)} posts on {url}")

            for pid in sorted(pids, key=lambda x: int(x)):
                if pid in seen:
                    continue
                try:
                    thread_meta = parse_thread_page(pid); sleep()
                    post_json   = get_post_json(pid);     sleep()

                    rec = normalize_record(pid, thread_meta, post_json,
                                           origin_tag=tag, order_variant=order_name)
                    if rec:
                        jf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        jf.flush()
                        collected_total += 1
                        term_collected += 1
                        seen.add(pid)

                        if collected_total % 25 == 0:
                            print(f"Collected {collected_total} items so far...")
                    else:
                        if not (thread_meta.get("question_text") or "").strip():
                            drop_stats["empty_question"] += 1
                        elif not (thread_meta.get("best_answer") or {}).get("text"):
                            drop_stats["no_answer"] += 1
                        else:
                            drop_stats["excluded_or_other"] += 1
                except requests.HTTPError as e:
                    print(f"[warn] HTTP error pid={pid}: {e}")
                    sleep()
                except Exception as e:
                    print(f"[warn] failed pid={pid}: {e}")
                    sleep()

            save_seen_ids(seen)
            sleep()

        print(f"[{tag}] collected={term_collected}")

print(f"\nDone. Collected {collected_total} items → {JSONL_PATH}")
print("Drop stats:", dict(drop_stats))



=== TAG: fastq ===
[fastq|creation] 69 posts on https://www.biostars.org/tag/fastq/?order=creation
Collected 25 items so far...
[fastq|rank] 69 posts on https://www.biostars.org/tag/fastq/?order=rank
Collected 50 items so far...
[fastq|votes] 69 posts on https://www.biostars.org/tag/fastq/?order=votes
Collected 75 items so far...
[fastq|replies] 69 posts on https://www.biostars.org/tag/fastq/?order=replies
Collected 100 items so far...
[fastq] collected=119

Done. Collected 119 items → ../04-output/raw/biostars/biostars_metagenomics_tag.jsonl
Drop stats: {'no_answer': 82}


In [10]:
# ========================
# Emit CSV snapshot
# ========================
field_order = [
    "source","post_id","url","title","question","answer","accepted",
    "score","tags_site","created_at","has_accepted","view_count","topic_labels",
    "search_query","crawl_tag","crawl_order"
]

rows = []
if JSONL_PATH.exists():
    with open(JSONL_PATH, "r", encoding="utf-8") as jf:
        for line in jf:
            obj = json.loads(line)
            obj["topic_labels"] = ";".join(obj.get("topic_labels", []))
            tags = obj.get("tags_site", [])
            if isinstance(tags, list):
                obj["tags_site"] = ";".join(tags)
            rows.append({k: obj.get(k, "") for k in field_order})

    with open(CSV_PATH, "w", encoding="utf-8", newline="") as cf:
        w = csv.DictWriter(cf, fieldnames=field_order)
        w.writeheader()
        for r in rows:
            w.writerow(r)

    print(f"Wrote CSV: {CSV_PATH} (rows={len(rows)})")
else:
    print(f"No JSONL at {JSONL_PATH}, skipping CSV.")

Wrote CSV: ../04-output/raw/biostars/biostars_metagenomics_tag.csv (rows=119)
